In [0]:
databricks notes

In [0]:
# Databricks notebook source
#  %md
#  # Delta Table Basics
# 
#  ## Topics
#  - Unity Catalog: Catalog → Schema → Table
#  - Create a Delta table
#  - Read Delta tables with PySpark
#  - Read Delta tables with Spark SQL
#  - Write DataFrames to Delta tables
#  - Append and overwrite
#  - Basic table permissions

#Create schema and table

In [0]:
# Create schema and Delta table
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.my_databricks_demo
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.my_databricks_demo.transactions (
    transaction_id STRING,
    customer_id STRING,
    product_id STRING,
    store_id STRING,
    transaction_date DATE,
    quantity INT,
    unit_price DOUBLE,
    total_amount DOUBLE
)
USING DELTA
""")

print("Schema and Delta table created successfully.")

#Insert  data

In [0]:
from datetime import date

data = [
    ("T001", "C001", "P001", "S001", date(2026, 8, 1), 2, 10.50, 21.00),
    ("T002", "C002", "P002", "S002", date(2026, 8, 2), 1, 25.00, 25.00),
    ("T003", "C003", "P003", "S001", date(2026, 8, 3), 3, 15.00, 45.00),
    ("T004", "C004", "P001", "S003", date(2026, 8, 4), 2, 50.00, 100.00),
    ("T005", "C005", "P004", "S002", date(2026, 8, 5), 1, 75.00, 75.00)
]

columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "store_id",
    "transaction_date",
    "quantity",
    "unit_price",
    "total_amount"
]

df = spark.createDataFrame(data, columns)

display(df)

#Write DataFrame to Delta

In [0]:
(
    df.write
      .format("delta")
      .mode("append")
      .saveAsTable("workspace.my_databricks_demo.transactions")
)

print("Data written to Delta table.")

#Read with PySpark

In [0]:
df_transactions = spark.read.table(
    "workspace.my_databricks_demo.transactions"
)

df_transactions.printSchema()

display(df_transactions)

#Filter using PySpark

In [0]:
display(
    df_transactions.filter(df_transactions.total_amount > 50).select( "transaction_id","transaction_date","total_amount" )
)

#Read using Spark SQL

In [0]:
display(
    spark.sql("""
        SELECT
            store_id,
            COUNT(*) AS transaction_count,
            SUM(total_amount) AS total_sales
        FROM workspace.my_databricks_demo.transactions
        GROUP BY store_id
        ORDER BY total_sales DESC
    """)
)

In [0]:
%sql

   SELECT
            store_id,
            COUNT(*) AS transaction_count,
            SUM(total_amount) AS total_sales
        FROM workspace.my_databricks_demo.transactions
        GROUP BY store_id
        ORDER BY total_sales DESC

#Convert SQL result back to Python DataFrame

In [0]:
sales_df = spark.sql("""
    SELECT
        transaction_id,
        transaction_date,
        total_amount
    FROM workspace.my_databricks_demo.transactions
    WHERE total_amount >= 25
""")

display(sales_df)

In [0]:
from pyspark.sql.functions import col

discounted_df = sales_df.withColumn(
    "discounted_amount",
    col("total_amount") * 0.90
)

display(discounted_df)

#Append more data

In [0]:
new_data = [
    ("T006", "C006", "P005", "S001", date(2026, 8, 6), 2, 30.00, 60.00),
    ("T007", "C007", "P006", "S003", date(2026, 8, 7), 1, 120.00, 120.00)
]

new_df = spark.createDataFrame(new_data, columns)

(
    new_df.write
          .format("delta")
          .mode("append")
          .saveAsTable("workspace.my_databricks_demo.transactions")
)

print("New rows appended.")

Check count

In [0]:
print(spark.table( "workspace.my_databricks_demo.transactions" ).count())

#Demonstrate overwrite

In [0]:
overwrite_data = [
    ("T008", "C008", "P007", "S002", date(2026, 8, 8), 1, 200.00, 200.00)
]

overwrite_df = spark.createDataFrame(overwrite_data, columns)

(
    overwrite_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(
                    "workspace.my_databricks_demo.transactions"
                )
)

display(
    spark.table(
        "workspace.my_databricks_demo.transactions"
    )
)

When we did:

overwrite_data = [
    ("T008", "C008", "P007", "S002", date(2026, 8, 8), 1, 200.00, 200.00)
]

overwrite_df = spark.createDataFrame(overwrite_data, columns)

Spark infers the Python values' types.

For example:

1        → integer/long type
200.00   → double

Depending on the Spark/Python environment, an integer created from Python can become LongType (BIGINT) rather than IntegerType (INT).

So you can end up with:

Existing Delta table
quantity → INT

while the new DataFrame has:

overwrite_df
quantity → BIGINT/LONG

Even though both are called:

quantity

Delta says:

"I have a quantity column already, but the incoming quantity has a different type."

Hence:

DELTA_FAILED_TO_MERGE_FIELDS
Failed to merge fields 'quantity' and 'quantity'
Why did our first table have INT?

Because we explicitly created it:

quantity INT

So:

Delta table:
quantity → INT

But when we later did:

spark.createDataFrame(overwrite_data, columns)

we allowed Spark to infer the DataFrame schema from Python objects.

That's exactly why explicit schemas matter

For production data engineering, we don't want:

In [0]:
# ============================================================
# CHECK THE SCHEMA
# ============================================================

print("Existing Delta table schema:")
spark.table(
    "workspace.my_databricks_demo.transactions"
).printSchema()

print("Overwrite DataFrame schema:")
overwrite_df.printSchema()

#Inspect the table

In [0]:
spark.sql("""
DESCRIBE TABLE workspace.my_databricks_demo.transactions
""").show(truncate=False)

#Table history

In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY
        workspace.my_databricks_demo.transactions
    """)
)

In [0]:
# ============================================================
# MODULE 1: DELTA TABLE BASICS 
# ============================================================
# Covered Topics:
# - Unity Catalog
# - Schema and table creation
# - Delta tables
# - Reading with PySpark
# - Reading with Spark SQL
# - Writing DataFrames
# - Append and overwrite
# - Table history